# Ask-M Live Inference (Kaggle)

• Base: google/gemma-3-12b-it (4-bit)
• LoRAs: Exam + Guided
• Backend-compatible FastAPI


In [ ]:
!pip install -U transformers peft accelerate bitsandbytes fastapi uvicorn nest-asyncio

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from fastapi import FastAPI
import uvicorn
import nest_asyncio

In [9]:
BASE_MODEL = "unsloth/gemma-3-12b-it-bnb-4bit"
EXAM_LORA  = "walterwhite91/ask-m-gemma3-exam-lora"
GUIDED_LORA = "walterwhite91/ask-m-gemma3-guide-lora"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from huggingface_hub import login
login(token="hf ko token")

In [11]:
#Load tokenizer + base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()
print("Base model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Base model loaded


In [ ]:
model = PeftModel.from_pretrained(
    model,
    EXAM_LORA,
    adapter_name="exam"
)

model.load_adapter(
    GUIDED_LORA,
    adapter_name="guided"
)

print("Exam + Guided LoRAs loaded")

In [13]:
def build_exam_prompt(subject: str, marks: int, question: str) -> str:
    return (
        "You are Ask-M, a strict exam-writing assistant.\n"
        f"Subject: {subject}\n"
        f"Marks: {marks}\n\n"
        "Write the best possible exam answer.\n"
        "Rules:\n"
        "- Keep it concise and mark-oriented.\n"
        "- Use correct definitions, key points, and formulas if relevant.\n"
        "- Use simple, clear wording.\n"
        "- Avoid extra fluff.\n\n"
        "Question:\n"
        f"{question}\n\n"
        "Answer:\n"
    )

def build_guided_prompt(subject: str, question: str) -> str:
    return (
        "You are Ask-M in guided/teaching mode.\n"
        f"Subject: {subject}\n\n"
        "Teach the student step-by-step.\n"
        "Rules:\n"
        "- Explain concepts simply.\n"
        "- Use analogies or small examples when helpful.\n"
        "- End with a short summary + 2 quick practice questions.\n\n"
        "Question:\n"
        f"{question}\n\n"
        "Guidance:\n"
    )

In [14]:
@torch.no_grad()
def run_inference(question, mode="exam", subject="General", marks=4):
    if mode == "guided":
        model.set_adapter("guided")
        prompt = build_guided_prompt(subject, question)
    else:
        model.set_adapter("exam")
        prompt = build_exam_prompt(subject, marks, question)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

In [15]:
app = FastAPI()

@app.post("/infer")
def infer(payload: dict):
    try:
        answer = run_inference(
            question=payload["question"],
            mode=payload.get("mode", "exam"),
            subject=payload.get("subject", "General"),
            marks=payload.get("marks", 4),
        )
        return {"status": "ok", "answer": answer}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8001,
        log_level="info"
    )

uvicorn_thread = threading.Thread(target=run_uvicorn, daemon=True)
uvicorn_thread.start()

print("Uvicorn running on port 8001")


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

In [27]:
!chmod +x cloudflared-linux-amd64


In [28]:
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared


In [29]:
!cloudflared --version

cloudflared version 2026.2.0 (built 2026-02-06-14:47 UTC)


In [ ]:
!cloudflared tunnel --url http://localhost:8001